In [3]:
"""
=============================================================
  CRYPTOVAULT - Final Project
  Capital University of Science & Technology
=============================================================
  HOW TO RUN:
      Put these 3 files in ONE folder:
        1. cryptovault_final.py   <-- run this
        2. cryptofunctions.py     <-- teammate's crypto code
        3. dsa_module.py          <-- your DSA code

      Then run:   python cryptovault_final.py

=============================================================
  THREE PARTS IN THIS FILE:
    PART 1 - FILE HANDLING    (Turab Fatima)
    PART 2 - DSA STRUCTURES   (Waaiz Waseem) - imported from dsa_module.py
    PART 3 - CRYPTOGRAPHY     (Amna Rafaqat) - imported from cryptofunctions.py
=============================================================
"""

import os
import shutil
import datetime

# ── Import teammate's crypto module (Part 3) ──────────────────
from cryptofunctions import (
    caesar_encrypt,
    caesar_decrypt,
    transposition_encrypt,
    transposition_decrypt,
    aes_encrypt_file,
    aes_decrypt_file,
    generate_sha256,
    generate_md5,
    save_hash_to_file,
    verify_file_integrity_from_hash_file,
    diffie_hellman_caesar_shift,
    check_openssl_installed,
    CryptoError,
)

# ── Import DSA module (Part 2 - your part) ────────────────────
from dsa_module import (
    LinkedList,
    UndoStack,
    BatchQueue,
    merge_sort,
    binary_search_by_id,
    linear_search,
)


# =============================================================
#  PART 1 - FILE HANDLING
#  Manages workspace folders, reading/writing/deleting files,
#  and a file registry (dictionary tracking all files)
# =============================================================

# Folder names used throughout the project
WORKSPACE   = "CryptoVault"
FOLDER_FILES     = os.path.join(WORKSPACE, "files")
FOLDER_ENCRYPTED = os.path.join(WORKSPACE, "encrypted")
FOLDER_DECRYPTED = os.path.join(WORKSPACE, "decrypted")
FOLDER_HASHES    = os.path.join(WORKSPACE, "hashes")
FOLDER_LOGS      = os.path.join(WORKSPACE, "logs")

# File registry - a dictionary that tracks every file created
# Format: { "filename": {"path": ..., "size": ..., "created": ..., "last_action": ...} }
file_registry = {}


def setup_workspace():
    """Create all workspace folders if they don't exist already."""
    for folder in [FOLDER_FILES, FOLDER_ENCRYPTED, FOLDER_DECRYPTED,
                   FOLDER_HASHES, FOLDER_LOGS]:
        os.makedirs(folder, exist_ok=True)


def create_new_file(filename, content):
    #create a new text file in the files/ folder.
    #also adds it to the file_registry dictionary.
    path = os.path.join(FOLDER_FILES, filename)

    if os.path.exists(path):
        print(f"  Error: '{filename}' already exists.")
        return None

    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

    # Add to registry dictionary
    file_registry[filename] = {
        "path"        : path,
        "size"        : len(content),
        "created"     : get_timestamp(),
        "last_action" : "CREATED"
    }

    return path


def read_file(path):
    #read and return content of a text file.
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def write_file(path, content):
    #write content to a file (creates folders if needed).
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)


def delete_file(folder, filename):
    #Delete a file from a folder and remove from registry.
    path = os.path.join(folder, filename)
    if not os.path.exists(path):
        print(f"  Error: '{filename}' not found.")
        return False
    os.remove(path)
    if filename in file_registry:
        del file_registry[filename]
    return True


def list_files(folder):
    """Return list of files inside a folder."""
    if not os.path.exists(folder):
        return []
    return sorted(os.listdir(folder))


def show_files(folder, folder_label):
    """Print all files in a folder with their sizes."""
    files = list_files(folder)
    print(f"\n  Files in {folder_label}/:")
    if not files:
        print("  (empty)")
        return
    print(f"  {'Name':<35} Size")
    print("  " + "-" * 50)
    for fname in files:
        fpath = os.path.join(folder, fname)
        size  = os.path.getsize(fpath)
        print(f"  {fname:<35} {size} bytes")


def preview_file(folder, filename):
    """Show first 600 characters of a file."""
    path = os.path.join(folder, filename)
    if not os.path.exists(path):
        print(f"  File not found: {filename}")
        return
    size = os.path.getsize(path)
    print(f"\n  --- {filename} ({size} bytes) ---")
    try:
        content = read_file(path)
        print(content[:600])
        if len(content) > 600:
            print(f"\n  ... ({len(content) - 600} more characters)")
    except UnicodeDecodeError:
        # Binary file (like AES encrypted) - show note instead
        print("  [Encrypted binary file - cannot display as text]")


def show_registry():
    """Display the file registry dictionary."""
    if not file_registry:
        print("\n  Registry is empty. Create some files first.")
        return
    print(f"\n  {'Filename':<28} {'Size':<8} {'Created':<22} Last Action")
    print("  " + "-" * 75)
    for name, info in file_registry.items():
        print(f"  {name:<28} {str(info['size'])+'B':<8} "
              f"{info['created']:<22} {info['last_action']}")


def update_registry(filename, action):
    """Update the last_action for a file in the registry."""
    if filename in file_registry:
        file_registry[filename]["last_action"] = action


def get_timestamp():
    """Return current date-time as a string."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def get_timestamp_short():
    """Return timestamp for use in filenames."""
    return datetime.datetime.now().strftime("%Y%m%d_%H%M%S")


def pick_folder():
    """Ask user to choose a folder and return its path."""
    print("\n  Which folder?")
    print("  [1] files      (original files)")
    print("  [2] encrypted  (encrypted files)")
    print("  [3] decrypted  (decrypted files)")
    print("  [4] hashes     (fingerprint files)")
    choice = input("  > ").strip()
    folders = {
        "1": FOLDER_FILES,
        "2": FOLDER_ENCRYPTED,
        "3": FOLDER_DECRYPTED,
        "4": FOLDER_HASHES,
    }
    return folders.get(choice, FOLDER_FILES)


# =============================================================
#  GLOBAL DSA OBJECTS (Part 2 - your structures)
#  These are created once and used throughout the whole program
# =============================================================

history = LinkedList()   # Linked List  - logs every operation
undo    = UndoStack()    # Stack (LIFO) - enables undo
batch   = BatchQueue()   # Queue (FIFO) - batch processing


def log_operation(action, filename, algorithm="N/A",
                  status="SUCCESS", output_path="", extra=""):
    """
    Called after every operation.
    - Appends to LinkedList (history log)
    - Pushes to UndoStack (for undo)
    - Updates file registry
    """
    history.append(action, filename, algorithm, status)
    if status == "SUCCESS":
        undo.push(action, filename, output_path, algorithm, extra)
    update_registry(filename, action)


# =============================================================
#  MENU HELPERS
# =============================================================

def pause():
    input("\nPress Enter to continue...")


def print_header():
    print()
    print("=" * 55)
    print("         CRYPTOVAULT - Secure File Toolkit")
    print("=" * 55)


def print_status():
    print(f"  Log entries : {history.count()}")
    print(f"  Batch queue : {batch.size()} job(s)")
    print(f"  Undo stack  : {undo.size()} action(s)")


# =============================================================
#  MAIN MENU
# =============================================================

def main_menu():
    setup_workspace()
    while True:
        print_header()
        print_status()
        print()
        print("  [1] Workspace")
        print("  [2] Encrypt Files")
        print("  [3] Decrypt Files")
        print("  [4] File Integrity (Hashing)")
        print("  [5] Batch Operations")
        print("  [6] Activity Log")
        print("  [7] Search and Reports")
        print("  [8] Settings")
        print("  [9] Exit")
        print()
        choice = input("  cryptovault> ").strip()

        if   choice == "1": workspace_menu()
        elif choice == "2": encrypt_menu()
        elif choice == "3": decrypt_menu()
        elif choice == "4": integrity_menu()
        elif choice == "5": batch_menu()
        elif choice == "6": activity_log_menu()
        elif choice == "7": search_menu()
        elif choice == "8": settings_menu()
        elif choice == "9":
            print("\n  Goodbye!")
            break
        else:
            print("  Invalid choice. Enter a number from 1 to 9.")


# =============================================================
#  [1] WORKSPACE MENU - File Handling operations
# =============================================================

def workspace_menu():
    while True:
        print()
        print("  --- WORKSPACE ---")
        print("  [1] Create new text file")
        print("  [2] View original files")
        print("  [3] View encrypted files")
        print("  [4] View decrypted files")
        print("  [5] Preview a file")
        print("  [6] Delete a file")
        print("  [7] View file registry")
        print("  [8] Back")
        print()
        choice = input("  workspace> ").strip()

        if choice == "1":
            filename = input("  File name (e.g. notes.txt): ").strip()
            if not filename:
                print("  Name cannot be empty.")
                pause()
                continue
            print("  Type your content. When done, type END on a new line:")
            lines = []
            while True:
                line = input()
                if line.strip().upper() == "END":
                    break
                lines.append(line)
            content = "\n".join(lines)
            path = create_new_file(filename, content)
            if path:
                print(f"\n  File created: {path}")
                log_operation("CREATE", filename, output_path=path)
            pause()

        elif choice == "2":
            show_files(FOLDER_FILES, "files")
            pause()

        elif choice == "3":
            show_files(FOLDER_ENCRYPTED, "encrypted")
            pause()

        elif choice == "4":
            show_files(FOLDER_DECRYPTED, "decrypted")
            pause()

        elif choice == "5":
            folder = pick_folder()
            filename = input("  File name: ").strip()
            preview_file(folder, filename)
            pause()

        elif choice == "6":
            folder = pick_folder()
            filename = input("  File name to delete: ").strip()
            confirm = input(f"  Delete '{filename}'? (yes/no): ").strip().lower()
            if confirm == "yes":
                if delete_file(folder, filename):
                    print(f"  Deleted: {filename}")
                    log_operation("DELETE", filename)
            else:
                print("  Cancelled.")
            pause()

        elif choice == "7":
            show_registry()
            pause()

        elif choice == "8":
            break
        else:
            print("  Invalid choice.")


# =============================================================
#  [2] ENCRYPT MENU - uses cryptofunctions.py (Amna's part)
# =============================================================

def encrypt_menu():
    while True:
        print()
        print("  --- ENCRYPT FILES ---")
        print("  [1] Caesar Cipher")
        print("  [2] Transposition Cipher")
        print("  [3] AES / OpenSSL")
        print("  [4] Diffie-Hellman + Caesar")
        print("  [5] Back")
        print()
        choice = input("  encrypt> ").strip()

        if choice == "1":
            filename = input("  File name (in files/ folder): ").strip()
            try:
                shift    = int(input("  Shift key: "))
                in_path  = os.path.join(FOLDER_FILES, filename)
                out_name = filename.replace(".txt", "_caesar_enc.txt")
                out_path = os.path.join(FOLDER_ENCRYPTED, out_name)
                content  = read_file(in_path)
                write_file(out_path, caesar_encrypt(content, shift))
                print(f"\n  Done! Saved to: {out_path}")
                log_operation("ENCRYPT", filename, "Caesar",
                              output_path=out_path, extra=str(shift))
            except Exception as e:
                print(f"  Error: {e}")
                log_operation("ENCRYPT", filename, "Caesar", status="FAILED")
            pause()

        elif choice == "2":
            filename = input("  File name (in files/ folder): ").strip()
            try:
                key      = int(input("  Column key (must be > 1): "))
                in_path  = os.path.join(FOLDER_FILES, filename)
                out_name = filename.replace(".txt", "_transposition_enc.txt")
                out_path = os.path.join(FOLDER_ENCRYPTED, out_name)
                content  = read_file(in_path)
                write_file(out_path, transposition_encrypt(content, key))
                print(f"\n  Done! Saved to: {out_path}")
                log_operation("ENCRYPT", filename, "Transposition",
                              output_path=out_path, extra=str(key))
            except Exception as e:
                print(f"  Error: {e}")
                log_operation("ENCRYPT", filename, "Transposition", status="FAILED")
            pause()

        elif choice == "3":
            filename = input("  File name (in files/ folder): ").strip()
            try:
                password = input("  Password: ")
                in_path  = os.path.join(FOLDER_FILES, filename)
                out_name = filename.replace(".txt", "_aes.enc")
                out_path = os.path.join(FOLDER_ENCRYPTED, out_name)
                aes_encrypt_file(in_path, out_path, password)
                print(f"\n  Done! Saved to: {out_path}")
                log_operation("ENCRYPT", filename, "AES-256",
                              output_path=out_path)
            except CryptoError as e:
                print(f"  Crypto Error: {e}")
                log_operation("ENCRYPT", filename, "AES-256", status="FAILED")
            except Exception as e:
                print(f"  Error: {e}")
                log_operation("ENCRYPT", filename, "AES-256", status="FAILED")
            pause()

        elif choice == "4":
            filename = input("  File name (in files/ folder): ").strip()
            try:
                p  = int(input("  Prime number p  : "))
                g  = int(input("  Primitive root g: "))
                ka = int(input("  Alice private key: "))
                kb = int(input("  Bob private key  : "))
                shift    = diffie_hellman_caesar_shift(p, g, ka, kb)
                in_path  = os.path.join(FOLDER_FILES, filename)
                out_name = filename.replace(".txt", "_dh_enc.txt")
                out_path = os.path.join(FOLDER_ENCRYPTED, out_name)
                content  = read_file(in_path)
                write_file(out_path, caesar_encrypt(content, shift))
                print(f"\n  Shared shift key : {shift}")
                print(f"  Saved to         : {out_path}")
                log_operation("ENCRYPT", filename, "DH-Caesar",
                              output_path=out_path, extra=str(shift))
            except Exception as e:
                print(f"  Error: {e}")
                log_operation("ENCRYPT", filename, "DH-Caesar", status="FAILED")
            pause()

        elif choice == "5":
            break
        else:
            print("  Invalid choice.")


# =============================================================
#  [3] DECRYPT MENU
# =============================================================

def decrypt_menu():
    while True:
        print()
        print("  --- DECRYPT FILES ---")
        print("  [1] Caesar Cipher")
        print("  [2] Transposition Cipher")
        print("  [3] AES / OpenSSL")
        print("  [4] Back")
        print()
        choice = input("  decrypt> ").strip()

        if choice == "1":
            filename = input("  Encrypted file name (in encrypted/ folder): ").strip()
            try:
                shift    = int(input("  Shift key used during encryption: "))
                in_path  = os.path.join(FOLDER_ENCRYPTED, filename)
                out_name = filename.replace("_enc", "_dec")
                out_path = os.path.join(FOLDER_DECRYPTED, out_name)
                content  = read_file(in_path)
                write_file(out_path, caesar_decrypt(content, shift))
                print(f"\n  Done! Saved to: {out_path}")
                log_operation("DECRYPT", filename, "Caesar", output_path=out_path)
            except Exception as e:
                print(f"  Error: {e}")
                log_operation("DECRYPT", filename, "Caesar", status="FAILED")
            pause()

        elif choice == "2":
            filename = input("  Encrypted file name (in encrypted/ folder): ").strip()
            try:
                key      = int(input("  Column key used during encryption: "))
                in_path  = os.path.join(FOLDER_ENCRYPTED, filename)
                out_name = filename.replace("_enc", "_dec")
                out_path = os.path.join(FOLDER_DECRYPTED, out_name)
                content  = read_file(in_path)
                write_file(out_path, transposition_decrypt(content, key))
                print(f"\n  Done! Saved to: {out_path}")
                log_operation("DECRYPT", filename, "Transposition",
                              output_path=out_path)
            except Exception as e:
                print(f"  Error: {e}")
                log_operation("DECRYPT", filename, "Transposition", status="FAILED")
            pause()

        elif choice == "3":
            filename = input("  Encrypted file name (e.g. notes_aes.enc): ").strip()
            try:
                password = input("  Password: ")
                in_path  = os.path.join(FOLDER_ENCRYPTED, filename)
                out_name = filename.replace(".enc", "_dec.txt")
                out_path = os.path.join(FOLDER_DECRYPTED, out_name)
                aes_decrypt_file(in_path, out_path, password)
                print(f"\n  Done! Saved to: {out_path}")
                log_operation("DECRYPT", filename, "AES-256", output_path=out_path)
            except CryptoError as e:
                print(f"  Crypto Error: {e}")
                log_operation("DECRYPT", filename, "AES-256", status="FAILED")
            except Exception as e:
                print(f"  Error: {e}")
                log_operation("DECRYPT", filename, "AES-256", status="FAILED")
            pause()

        elif choice == "4":
            break
        else:
            print("  Invalid choice.")


# =============================================================
#  [4] FILE INTEGRITY MENU - SHA-256 / MD5 hashing
# =============================================================

def integrity_menu():
    while True:
        print()
        print("  --- FILE INTEGRITY ---")
        print("  [1] Generate SHA-256 fingerprint")
        print("  [2] Generate MD5 fingerprint")
        print("  [3] Save fingerprint to file")
        print("  [4] Verify file against saved fingerprint")
        print("  [5] Compare MD5 vs SHA-256")
        print("  [6] Back")
        print()
        choice = input("  integrity> ").strip()

        if choice == "1":
            folder   = pick_folder()
            filename = input("  File name: ").strip()
            path     = os.path.join(folder, filename)
            try:
                h = generate_sha256(path)
                print(f"\n  SHA-256:\n  {h}")
                log_operation("HASH", filename, "SHA-256")
            except Exception as e:
                print(f"  Error: {e}")
            pause()

        elif choice == "2":
            folder   = pick_folder()
            filename = input("  File name: ").strip()
            path     = os.path.join(folder, filename)
            try:
                h = generate_md5(path)
                print(f"\n  MD5:\n  {h}")
                log_operation("HASH", filename, "MD5")
            except Exception as e:
                print(f"  Error: {e}")
            pause()

        elif choice == "3":
            folder    = pick_folder()
            filename  = input("  File name: ").strip()
            hash_type = input("  Hash type (sha256 / md5): ").strip().lower()
            file_path = os.path.join(folder, filename)
            hash_path = os.path.join(FOLDER_HASHES,
                                     filename + f"_{hash_type}.hash")
            try:
                h = save_hash_to_file(file_path, hash_path, hash_type)
                print(f"\n  Fingerprint saved to: {hash_path}")
                print(f"  {h}")
                log_operation("HASH_SAVE", filename, hash_type.upper(),
                              output_path=hash_path)
            except Exception as e:
                print(f"  Error: {e}")
            pause()

        elif choice == "4":
            folder        = pick_folder()
            filename      = input("  File name to verify: ").strip()
            hash_filename = input("  Hash file name (in hashes/ folder): ").strip()
            file_path     = os.path.join(folder, filename)
            hash_path     = os.path.join(FOLDER_HASHES, hash_filename)
            try:
                result = verify_file_integrity_from_hash_file(file_path, hash_path)
                print(f"\n  Stored hash  : {result['stored_hash']}")
                print(f"  Current hash : {result['current_hash']}")
                if result["is_valid"]:
                    print("  Status       : VALID - file has NOT been changed")
                else:
                    print("  Status       : INVALID - file has been MODIFIED")
                log_operation("VERIFY", filename, "SHA-256",
                              status="SUCCESS" if result["is_valid"] else "FAILED")
            except Exception as e:
                print(f"  Error: {e}")
            pause()

        elif choice == "5":
            folder   = pick_folder()
            filename = input("  File name: ").strip()
            path     = os.path.join(folder, filename)
            try:
                print(f"\n  MD5    : {generate_md5(path)}")
                print(f"  SHA-256: {generate_sha256(path)}")
                print("\n  SHA-256 is more secure. MD5 is shown for comparison only.")
                log_operation("HASH_COMPARE", filename, "MD5+SHA-256")
            except Exception as e:
                print(f"  Error: {e}")
            pause()

        elif choice == "6":
            break
        else:
            print("  Invalid choice.")


# =============================================================
#  [5] BATCH OPERATIONS - uses BatchQueue (your DSA part)
#  Queue = FIFO: first file added is first file processed
# =============================================================

def batch_menu():
    while True:
        print()
        print(f"  --- BATCH OPERATIONS ({batch.size()} jobs in queue) ---")
        print("  [1] Add file to queue")
        print("  [2] View all queued jobs")
        print("  [3] Remove next job from queue")
        print("  [4] Process all Caesar jobs")
        print("  [5] Process all Transposition jobs")
        print("  [6] Process all AES jobs")
        print("  [7] Process all SHA-256 hash jobs")
        print("  [8] Clear entire queue")
        print("  [9] Back")
        print()
        choice = input("  batch> ").strip()

        if choice == "1":
            filename = input("  File name (must be in files/ folder): ").strip()
            print("  Operation: [1] CAESAR  [2] TRANSPOSITION  [3] AES  [4] SHA256")
            op_choice = input("  > ").strip()
            op_map = {"1": "CAESAR", "2": "TRANSPOSITION",
                      "3": "AES",    "4": "SHA256"}
            operation = op_map.get(op_choice)
            if not operation:
                print("  Invalid operation.")
            else:
                batch.enqueue(filename, operation)
                print(f"  Added to queue: {operation} on '{filename}'")
                print(f"  Queue size is now: {batch.size()}")
            pause()

        elif choice == "2":
            batch.display()
            pause()

        elif choice == "3":
            removed = batch.remove_front()
            if removed:
                print(f"  Removed: {removed['operation']} on '{removed['file_name']}'")
            else:
                print("  Queue is empty.")
            pause()

        elif choice in ("4", "5", "6", "7"):
            op_map = {"4": "CAESAR", "5": "TRANSPOSITION",
                      "6": "AES",    "7": "SHA256"}
            run_batch(op_map[choice])

        elif choice == "8":
            batch.clear()
            print("  Queue cleared.")
            pause()

        elif choice == "9":
            break
        else:
            print("  Invalid choice.")


def run_batch(operation_type):
    """
    Pulls matching jobs from the queue and processes them.
    The crypto callback function is passed into process_all()
    so the BatchQueue stays independent of cryptofunctions.
    """
    # Separate matching jobs from others
    to_process = BatchQueue()
    keep_jobs  = BatchQueue()

    while not batch.is_empty():
        job = batch.dequeue()
        if job["operation"] == operation_type:
            to_process.enqueue(job["file_name"], job["operation"])
        else:
            keep_jobs.enqueue(job["file_name"], job["operation"])

    # Put non-matching jobs back in the main queue
    while not keep_jobs.is_empty():
        j = keep_jobs.dequeue()
        batch.enqueue(j["file_name"], j["operation"])

    if to_process.is_empty():
        print(f"\n  No {operation_type} jobs in queue.")
        pause()
        return

    # Ask for key/password once for all files
    shared_key = {}
    if operation_type == "CAESAR":
        shared_key["shift"] = int(input("  Caesar shift for all files: "))
    elif operation_type == "TRANSPOSITION":
        shared_key["key"] = int(input("  Column key for all files: "))
    elif operation_type == "AES":
        shared_key["password"] = input("  AES password for all files: ")

    # This callback is passed to BatchQueue.process_all()
    def process_one_job(job):
        fname   = job["file_name"]
        in_path = os.path.join(FOLDER_FILES, fname)
        try:
            if operation_type == "CAESAR":
                out_name = fname.replace(".txt", "_caesar_enc.txt")
                out_path = os.path.join(FOLDER_ENCRYPTED, out_name)
                write_file(out_path,
                           caesar_encrypt(read_file(in_path),
                                          shared_key["shift"]))
                log_operation("ENCRYPT", fname, "Caesar", output_path=out_path,
                              extra=str(shared_key["shift"]))

            elif operation_type == "TRANSPOSITION":
                out_name = fname.replace(".txt", "_transposition_enc.txt")
                out_path = os.path.join(FOLDER_ENCRYPTED, out_name)
                write_file(out_path,
                           transposition_encrypt(read_file(in_path),
                                                 shared_key["key"]))
                log_operation("ENCRYPT", fname, "Transposition",
                              output_path=out_path,
                              extra=str(shared_key["key"]))

            elif operation_type == "AES":
                out_name = fname.replace(".txt", "_aes.enc")
                out_path = os.path.join(FOLDER_ENCRYPTED, out_name)
                aes_encrypt_file(in_path, out_path, shared_key["password"])
                log_operation("ENCRYPT", fname, "AES-256", output_path=out_path)

            elif operation_type == "SHA256":
                out_path = os.path.join(FOLDER_HASHES, fname + "_sha256.hash")
                save_hash_to_file(in_path, out_path, "sha256")
                log_operation("HASH_SAVE", fname, "SHA-256", output_path=out_path)

            print(f"  Done: {fname}")
            return {"file": fname, "ok": True}

        except Exception as e:
            print(f"  Failed: {fname} - {e}")
            log_operation("ENCRYPT", fname, operation_type, status="FAILED")
            return {"file": fname, "ok": False}

    to_process.process_all(process_one_job)
    pause()


# =============================================================
#  [6] ACTIVITY LOG - uses LinkedList + UndoStack (your DSA)
# =============================================================

def activity_log_menu():
    while True:
        print()
        print(f"  --- ACTIVITY LOG ({history.count()} entries) ---")
        print("  [1] View recent activity (last 10)")
        print("  [2] View full log")
        print("  [3] Save log to file")
        print("  [4] Undo last operation")
        print("  [5] Back")
        print()
        choice = input("  activity> ").strip()

        if choice == "1":
            print("\n  Last 10 operations:")
            history.display(limit=10)
            pause()

        elif choice == "2":
            print(f"\n  All {history.count()} operations:")
            history.display()
            pause()

        elif choice == "3":
            log_path = os.path.join(FOLDER_LOGS,
                                    f"log_{get_timestamp_short()}.txt")
            try:
                history.save_to_file(log_path)
                print(f"\n  Log saved to: {log_path}")
            except Exception as e:
                print(f"  Error: {e}")
            pause()

        elif choice == "4":
            if undo.is_empty():
                print("\n  Nothing to undo.")
            else:
                entry     = undo.pop()
                action    = entry["action"]
                filename  = entry["file_name"]
                out_path  = entry.get("output_path", "")
                print(f"\n  Undoing: {action} on '{filename}'")

                # Delete the output file to reverse the operation
                if action in ("ENCRYPT", "DECRYPT", "HASH_SAVE"):
                    if out_path and os.path.exists(out_path):
                        os.remove(out_path)
                        print(f"  Deleted: {out_path}")
                    else:
                        print("  Output file already removed.")

                elif action == "CREATE":
                    path = os.path.join(FOLDER_FILES, filename)
                    if os.path.exists(path):
                        os.remove(path)
                        print(f"  Deleted created file: {path}")

                # Log the undo itself
                history.append("UNDO", filename, entry["algorithm"])
            pause()

        elif choice == "5":
            break
        else:
            print("  Invalid choice.")


# =============================================================
#  [7] SEARCH AND REPORTS - uses MergeSort + Search (your DSA)
# =============================================================

def search_menu():
    while True:
        print()
        print("  --- SEARCH AND REPORTS ---")
        print("  [1] Search by file name       (linear search)")
        print("  [2] Search by operation type  (linear search)")
        print("  [3] Search by algorithm       (linear search)")
        print("  [4] Search by status          (linear search)")
        print("  [5] Find by operation ID      (binary search)")
        print("  [6] Show sorted by file name  (merge sort)")
        print("  [7] Show sorted by algorithm  (merge sort)")
        print("  [8] Show sorted by timestamp  (merge sort)")
        print("  [9] Export report")
        print("  [10] Back")
        print()
        choice = input("  search> ").strip()

        if choice == "1":
            do_linear_search("file_name", "file name (e.g. notes.txt)")
        elif choice == "2":
            do_linear_search("action", "operation (e.g. ENCRYPT, DECRYPT, HASH)")
        elif choice == "3":
            do_linear_search("algorithm", "algorithm (e.g. Caesar, AES-256)")
        elif choice == "4":
            do_linear_search("status", "status (SUCCESS or FAILED)")
        elif choice == "5":
            do_binary_search()
        elif choice == "6":
            show_sorted("file_name", "File Name")
        elif choice == "7":
            show_sorted("algorithm", "Algorithm")
        elif choice == "8":
            show_sorted("timestamp", "Timestamp")
        elif choice == "9":
            export_report()
        elif choice == "10":
            break
        else:
            print("  Invalid choice.")


def do_linear_search(field, label):
    """Search all log entries using linear search."""
    query   = input(f"  Search {label}: ").strip()
    records = history.to_list()
    matches = linear_search(records, query, field=field)

    if not matches:
        print(f"\n  No results found for '{query}'.")
    else:
        print(f"\n  Found {len(matches)} result(s):")
        print_records(matches)
    pause()


def do_binary_search():
    """Find a log entry by ID using binary search (faster than linear)."""
    try:
        target_id = int(input("  Enter operation ID number: ").strip())
    except ValueError:
        print("  Please enter a valid number.")
        pause()
        return

    # First sort by ID (merge sort), then binary search
    sorted_records = merge_sort(history.to_list(), key="operation_id")
    result         = binary_search_by_id(sorted_records, target_id)

    if result:
        print(f"\n  Found:")
        print(f"  ID        : {result.operation_id}")
        print(f"  Action    : {result.action}")
        print(f"  File      : {result.file_name}")
        print(f"  Algorithm : {result.algorithm}")
        print(f"  Status    : {result.status}")
        print(f"  Time      : {result.timestamp}")
    else:
        print(f"\n  No record with ID {target_id} found.")
    pause()


def show_sorted(key, label):
    """Sort all log entries using merge sort and display them."""
    records = merge_sort(history.to_list(), key=key)
    if not records:
        print("\n  No log entries yet.")
        pause()
        return
    print(f"\n  Records sorted by {label}:")
    print_records(records)
    pause()


def print_records(records):
    """Print a list of log records in a neat table."""
    print(f"\n  {'ID':<5} {'Action':<12} {'Algorithm':<16} "
          f"{'Status':<10} {'File':<25} Time")
    print("  " + "-" * 85)
    for r in records:
        print(f"  {r.operation_id:<5} {r.action:<12} {r.algorithm:<16} "
              f"{r.status:<10} {r.file_name:<25} {r.timestamp}")


def export_report():
    """Export the full log to a text file, sorted by timestamp."""
    records  = merge_sort(history.to_list(), key="timestamp")
    out_path = os.path.join(FOLDER_LOGS, f"report_{get_timestamp_short()}.txt")
    os.makedirs(FOLDER_LOGS, exist_ok=True)

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("CryptoVault - Activity Report\n")
        f.write(f"Generated: {get_timestamp()}\n")
        f.write("=" * 85 + "\n")
        f.write(f"{'ID':<5} {'Action':<12} {'Algorithm':<16} "
                f"{'Status':<10} {'File':<25} Time\n")
        f.write("-" * 85 + "\n")
        for r in records:
            f.write(f"{r.operation_id:<5} {r.action:<12} {r.algorithm:<16} "
                    f"{r.status:<10} {r.file_name:<25} {r.timestamp}\n")

    print(f"\n  Report saved to: {out_path}")
    pause()


# =============================================================
#  [8] SETTINGS MENU
# =============================================================

def settings_menu():
    while True:
        print()
        print("  --- SETTINGS ---")
        print("  [1] Show workspace folders")
        print("  [2] Check OpenSSL")
        print("  [3] Show supported algorithms")
        print("  [4] Reset workspace (deletes everything)")
        print("  [5] Back")
        print()
        choice = input("  settings> ").strip()

        if choice == "1":
            print(f"\n  Workspace: {os.path.abspath(WORKSPACE)}")
            for folder in [FOLDER_FILES, FOLDER_ENCRYPTED,
                           FOLDER_DECRYPTED, FOLDER_HASHES, FOLDER_LOGS]:
                exists = "exists" if os.path.exists(folder) else "missing"
                print(f"  {folder:<40} [{exists}]")
            pause()

        elif choice == "2":
            try:
                check_openssl_installed()
                print("\n  OpenSSL is available.")
            except Exception as e:
                print(f"\n  OpenSSL not found: {e}")
            pause()

        elif choice == "3":
            print("\n  Supported algorithms:")
            print("  - Caesar Cipher")
            print("  - Transposition Cipher")
            print("  - AES-256-CBC (needs OpenSSL installed)")
            print("  - Diffie-Hellman key exchange -> Caesar shift")
            print("  - SHA-256 (file fingerprint)")
            print("  - MD5 (comparison only)")
            pause()

        elif choice == "4":
            confirm = input(
                "\n  This will DELETE all files. Type YES to confirm: "
            ).strip()
            if confirm == "YES":
                if os.path.exists(WORKSPACE):
                    shutil.rmtree(WORKSPACE)
                file_registry.clear()
                setup_workspace()
                print("  Workspace reset complete.")
                log_operation("RESET", "workspace")
            else:
                print("  Cancelled.")
            pause()

        elif choice == "5":
            break
        else:
            print("  Invalid choice.")


# =============================================================
#  START THE PROGRAM
# =============================================================

if __name__ == "__main__":
    main_menu()


         CRYPTOVAULT - Secure File Toolkit
  Log entries : 0
  Batch queue : 0 job(s)
  Undo stack  : 0 action(s)

  [1] Workspace
  [2] Encrypt Files
  [3] Decrypt Files
  [4] File Integrity (Hashing)
  [5] Batch Operations
  [6] Activity Log
  [7] Search and Reports
  [8] Settings
  [9] Exit



  cryptovault>  9



  Goodbye!
